[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap11_big_data/leitura_seletiva_agregacao_pyspark_sisdepen_2016.ipynb)

# Leitura seletiva e agregação com PySpark

Demonstração didática de leitura de uma planilha Excel, seleção de campos revisados, definição explícita de tipos e agregação por dimensões documentadas. A base 2016 do SISDEPEN contém 1.111 colunas; o exemplo limita a leitura a cinco campos e agrega totais de população prisional por UF.

O uso de Spark é apresentado para discutir a API e a lógica de processamento distribuído. Não se afirma que o tamanho desta base exige Spark, nem que o processamento apresente ganho de desempenho. Caminho sugerido no GitHub: `capitulos/cap11_big_data/leitura_seletiva_agregacao_pyspark_sisdepen_2016.ipynb`.

## 1. Questão analítica e esquema selecionado

O exemplo agrega os totais de população prisional masculina e feminina por unidade da Federação. A planilha contém uma linha por registro do estabelecimento penitenciário e campos de identificação, localização e total da população por sexo. As dimensões selecionadas são `UF:` e `Status`; a medida é a soma dos totais masculino e feminino. O status é mantido como dimensão para que registros incompletos ou não iniciados não sejam confundidos com registros completos.

Os campos foram selecionados após inspeção do cabeçalho do arquivo. Nomes das colunas quantitativas incluem espaços não separáveis, por isso são referenciados exatamente como aparecem. A leitura usa a URL fornecida pelo autor.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .appName("Leitura seletiva e agregacao SISDEPEN 2016")
         .master("local[*]")
         .getOrCreate())
spark.version

## 2. Instalação e obtenção do arquivo

No Google Colab, instale PySpark e um leitor de arquivos Excel. A leitura direta do XLSX por Spark depende de um conector compatível com a versão do Spark. Para manter o exemplo simples e reproduzível no notebook, a planilha é baixada e o pandas faz a leitura seletiva de cinco colunas; em seguida, os dados selecionados são convertidos para um DataFrame Spark. Essa etapa permite demonstrar transformação distribuída após a ingestão. Não se deve descrever esta estratégia como leitura nativa de XLSX pelo Spark.

In [ ]:
%pip -q install pyspark openpyxl

In [ ]:
import requests
from io import BytesIO
import pandas as pd

URL_ARQUIVO = "https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/2016_basefinal_depen_publicacao.xlsx"
resposta = requests.get(URL_ARQUIVO, timeout=60)
resposta.raise_for_status()
print("Bytes recebidos:", len(resposta.content))

COL_STATUS = "Status"
COL_UF = "UF:"
COL_MASC = "4.1.7.M. População prisional\xa0TOTAL | MASCULINO"
COL_FEM = "4.1.7.F População prisional\xa0TOTAL | FEMININO"
COLUNAS = [COL_STATUS, COL_UF, COL_MASC, COL_FEM]

pdf = pd.read_excel(BytesIO(resposta.content), sheet_name="Sheet1", usecols=COLUNAS)
print("Dimensões após leitura seletiva:", pdf.shape)
print("Campos lidos:", list(pdf.columns))
pdf.head()

## 3. Conversão para Spark e definição de tipos

Os campos categóricos são convertidos para texto e as medidas para inteiros longos, apropriados a contagens. Valores ausentes permanecem nulos nesta etapa. A conversão do DataFrame pandas para Spark é explícita, permitindo observar o esquema e a distribuição das partições. Como a fonte original é Excel, a carga da planilha em si não é distribuída nesta implementação.

In [ ]:
# Normalizar valores ausentes e tipos antes de transferir o conjunto selecionado ao Spark.
for c in [COL_STATUS, COL_UF]:
    pdf[c] = pdf[c].astype("string")
for c in [COL_MASC, COL_FEM]:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce").astype("Int64")

schema = T.StructType([
    T.StructField(COL_STATUS, T.StringType(), True),
    T.StructField(COL_UF, T.StringType(), True),
    T.StructField(COL_MASC, T.LongType(), True),
    T.StructField(COL_FEM, T.LongType(), True),
])
linhas = [tuple(None if pd.isna(v) else int(v) if c in [COL_MASC, COL_FEM] else str(v)
                for c, v in zip(COLUNAS, row)) for row in pdf[COLUNAS].itertuples(index=False, name=None)]
df = spark.createDataFrame(linhas, schema=schema)
df.printSchema()
df.show(5, truncate=False)
print("Partições iniciais:", df.rdd.getNumPartitions())

## 4. Validação dos registros e campos

Antes de agregar, verificam-se o número de linhas, valores ausentes nas dimensões, valores não numéricos convertidos para nulo e valores negativos nas medidas. A validação não transforma automaticamente nulos em zero: ausência de registro e valor zero têm significados diferentes. A interpretação do campo `Status` e da cobertura deve seguir a documentação da base.

In [ ]:
print("Linhas:", df.count())
print("Valores ausentes por coluna:")
df.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in COLUNAS]).show(truncate=False)
print("Valores negativos nas medidas:")
df.select(*[
    F.sum((F.col(c) < 0).cast("long")).alias(c) for c in [COL_MASC, COL_FEM]
]).show()
print("Categorias de status:")
df.groupBy(COL_STATUS).count().orderBy(F.desc("count")).show(truncate=False)
print("UFs distintas, incluindo nulos:", df.select(COL_UF).distinct().count())

## 5. Seleção, projeção e agregação

A API DataFrame expressa transformações como uma sequência de operações. A seleção de colunas reduz o esquema usado nas etapas seguintes; a conversão de tipos explicita a semântica das medidas; a agregação calcula totais por UF e status. A expressão `coalesce(..., 0)` abaixo é aplicada somente às medidas, para que a soma não seja nula quando uma das duas medidas está ausente. A tabela de saída também registra quantos registros contribuíram e quantas medidas estavam ausentes, apoiando a interpretação.

In [ ]:
base = (df
        .select(COL_STATUS, COL_UF,
                F.col(COL_MASC).cast("long").alias("populacao_masculina"),
                F.col(COL_FEM).cast("long").alias("populacao_feminina"))
        .withColumn("uf", F.trim(F.col(COL_UF)))
        .withColumn("status", F.trim(F.col(COL_STATUS))))

agregado = (base
    .groupBy("uf", "status")
    .agg(
        F.count(F.lit(1)).alias("registros"),
        F.sum(F.coalesce(F.col("populacao_masculina"), F.lit(0))).alias("total_masculino"),
        F.sum(F.coalesce(F.col("populacao_feminina"), F.lit(0))).alias("total_feminino"),
        F.sum(F.col("populacao_masculina").isNull().cast("long")).alias("ausentes_masculino"),
        F.sum(F.col("populacao_feminina").isNull().cast("long")).alias("ausentes_feminino")
    )
    .withColumn("total_populacao", F.col("total_masculino") + F.col("total_feminino"))
    .orderBy("uf", "status"))

agregado.show(40, truncate=False)
print("Linhas agregadas:", agregado.count())

## 6. Inspeção do plano lógico e do plano físico

O método `explain` apresenta etapas do plano de execução. O plano permite discutir projeção, agregação e operações de troca de dados entre partições, como *shuffle*, quando necessárias. A existência de uma etapa distribuída no plano não demonstra, por si só, ganho de desempenho. Neste exercício, a fonte é uma planilha pequena o suficiente para ser baixada e selecionada localmente; a finalidade é didática.

In [ ]:
agregado.explain(mode="formatted")

## 7. Comparação de consistência com uma agregação local

A conferência abaixo compara os totais por UF e status produzidos por Spark com uma agregação local da seleção em pandas. Essa verificação valida a lógica do exemplo para as colunas e regras especificadas. Ela não é um benchmark: não controla ambiente, leitura, cache, paralelismo, alocação de recursos ou repetições e, portanto, não sustenta conclusão sobre desempenho relativo.

In [ ]:
pdf_conf = pdf.copy()
pdf_conf[COL_UF] = pdf_conf[COL_UF].astype("string").str.strip()
pdf_conf[COL_STATUS] = pdf_conf[COL_STATUS].astype("string").str.strip()
pdf_conf[COL_MASC] = pd.to_numeric(pdf_conf[COL_MASC], errors="coerce").fillna(0)
pdf_conf[COL_FEM] = pd.to_numeric(pdf_conf[COL_FEM], errors="coerce").fillna(0)
pdf_conf["total_populacao"] = pdf_conf[COL_MASC] + pdf_conf[COL_FEM]
local = (pdf_conf.groupby([COL_UF, COL_STATUS], dropna=False)
         .agg(registros=(COL_STATUS, "size"),
              total_populacao=("total_populacao", "sum"))
         .reset_index())
spark_check = (agregado.select(F.col("uf").alias(COL_UF), F.col("status").alias(COL_STATUS), "registros", "total_populacao")
               .toPandas())
# Comparação numérica ordenada: categorias nulas são preservadas com marcador explícito.
for frame in [local, spark_check]:
    frame[COL_UF] = frame[COL_UF].fillna("[nulo]").astype(str)
    frame[COL_STATUS] = frame[COL_STATUS].fillna("[nulo]").astype(str)
local = local.rename(columns={COL_UF:"uf", COL_STATUS:"status"})
spark_check = spark_check.rename(columns={COL_UF:"uf", COL_STATUS:"status"})
comparacao = local.merge(spark_check, on=["uf","status"], suffixes=("_pandas","_spark"), how="outer")
comparacao["coincide"] = ((comparacao["registros_pandas"] == comparacao["registros_spark"]) &
                           (comparacao["total_populacao_pandas"] == comparacao["total_populacao_spark"]))
print("Todas as agregações conferem:", bool(comparacao["coincide"].all()))
comparacao.head(10)

## 8. Reflexão sobre execução distribuída

Em Spark, transformações como `select`, `withColumn` e `groupBy` compõem um plano que é avaliado quando uma ação, como `show`, `count` ou gravação, é solicitada. O ambiente pode particionar os dados e executar tarefas em paralelo, dependendo da configuração e dos recursos disponíveis. Operações de agregação por chave podem exigir redistribuição dos dados entre partições.

O exercício distingue a lógica da API da estratégia de ingestão: a planilha é lida de forma seletiva por pandas e o subconjunto é transformado em Spark. Para fontes em formatos distribuíveis, como Parquet, uma leitura nativa com `spark.read` permite aplicar a seleção no fluxo de leitura. Mesmo nesse cenário, a escolha de Spark deve considerar volume, complexidade, recursos e custos operacionais. Esta base e este exercício não demonstram que o uso de Spark seja necessário nem que seja mais rápido.

### Checklist

- Os campos selecionados foram revisados e seus nomes conferidos no arquivo?
- As dimensões da agregação, UF e Status, foram explicitadas?
- Os campos de população foram convertidos para tipos inteiros adequados?
- Nulos foram diferenciados de zeros e documentados?
- O total foi agregado sem misturar registros de status distintos?
- O resultado Spark foi conferido com uma agregação local?
- A explicação distingue leitura local da planilha e transformação em Spark?
- Não foram alegados necessidade do Spark ou ganho de desempenho a partir deste exemplo?